In [4]:
import spacy
import wikipediaapi

# 1. Load the scispaCy model
# This model identifies Diseases and Chemicals
print("Loading medical model...")
nlp = spacy.load("en_ner_bc5cdr_md")

# 2. Initialize Wikipedia API
wiki = wikipediaapi.Wikipedia(
    language='en',
    user_agent='MedicalGlossaryProject/1.0 (student_project)'
)

# --- THE NEW HELPER FUNCTION ---
def expand_term_for_search(term):
    """
    Checks if a term is a known abbreviation.
    Returns the full medical name if found, otherwise returns the original term.

    You can expand this dictionary based on the specific jargon in your dataset.
    """
    # Dictionary mapping abbreviations to their full Wikipedia-friendly names
    # Keys should be lowercase for easier matching
    abbreviations = {
        "mi": "Myocardial infarction",
        "iv": "Intravenous therapy",
        "bp": "Blood pressure",
        "cad": "Coronary artery disease",
        "copd": "Chronic obstructive pulmonary disease",
        "cns": "Central nervous system",
        "gi": "Gastrointestinal tract",
        "mg": "Milligram",
        "bid": "Twice a day", # Latin: bis in die
        "prn": "Pro re nata", # As needed
        "fx": "Bone fracture",
        "dx": "Diagnosis",
        "hx": "Medical history",
        "cv": "Cardiovascular system"
    }

    clean_term = term.lower().strip()

    # If the abbreviation exists in our dictionary, return the full name
    if clean_term in abbreviations:
        return abbreviations[clean_term]

    # Otherwise, return the original term
    return term

# --- DEFINITION FETCHER ---
def get_simple_definition(search_term):
    """
    Fetches the summary from Wikipedia using the SEARCH TERM
    (which might be different from the original text if we expanded it).
    """
    try:
        page = wiki.page(search_term)

        if page.exists():
            # Get first 2 sentences
            summary = page.summary.split('. ')[:2]
            return ". ".join(summary) + "."
        else:
            return "Definition not found."
    except Exception:
        return "Error retrieving definition."

# --- MAIN PIPELINE ---
def generate_glossary(text):
    print("Scanning text...")
    doc = nlp(text)

    seen_terms = set()
    glossary = {}

    for ent in doc.ents:
        original_word = ent.text.strip()

        # Only process if we haven't seen this exact word yet
        if original_word not in seen_terms:
            seen_terms.add(original_word)

            # 1. Expand abbreviation (e.g., "MI" -> "Myocardial infarction")
            search_term = expand_term_for_search(original_word)

            # 2. Fetch definition using the expanded term
            print(f"Defining: '{original_word}' (Searching as: '{search_term}')")
            definition = get_simple_definition(search_term)

            # 3. Store in glossary using the ORIGINAL word as the key
            glossary[original_word] = definition

    return glossary

# --- EXAMPLE RUN ---

text_data = """
PRIMARY OBJECTIVES: I. To determine the effects of the iron-chelating agent deferasirox on changes in: neutrophil function; macrophage function; lymphocyte function.

SECONDARY OBJECTIVES: I. To determine the effect of chelation on the incidence of bacterial, viral and fungal infections documented by clinical, microbiologically-proven versus radiologically-proven criteria. II. To determine the effect of iron chelation on mortality and morbidity with incidence of the following parameters: Need for hospitalization; Duration of hospitalization; Need for ventilatory support; Need for exchange transfusion/apheresis; Need for treatment with antifungals or antibiotics for documented infections.

OUTLINE: Patients receive oral deferasirox once daily for up to 6 months or until blood counts recover in the absence of disease progression or unacceptable toxicity.
"""

final_glossary = generate_glossary(text_data)

print("\n" + "="*30)
print("FINAL PATIENT GLOSSARY")
print("="*30)

for word, definition in final_glossary.items():
    print(f"\n🔹 WORD: {word}")
    print(f"   MEANING: {definition}")

Loading medical model...
Scanning text...
Defining: 'deferasirox' (Searching as: 'deferasirox')
Defining: 'fungal infections' (Searching as: 'fungal infections')
Defining: 'microbiologically-proven' (Searching as: 'microbiologically-proven')
Defining: 'iron' (Searching as: 'iron')
Defining: 'infections' (Searching as: 'infections')
Defining: 'toxicity' (Searching as: 'toxicity')

FINAL PATIENT GLOSSARY

🔹 WORD: deferasirox
   MEANING: Deferasirox, sold under the brand name Exjade among others, is an oral iron chelator. Its main use is to reduce chronic iron overload in patients who are receiving long-term blood transfusions for conditions such as beta-thalassemia and other chronic anemias.

🔹 WORD: fungal infections
   MEANING: Fungal infection, also known as mycosis, is a disease caused by fungi. Different types are traditionally divided according to the part of the body affected: superficial, subcutaneous, and systemic.

🔹 WORD: microbiologically-proven
   MEANING: Definition not fou

In [1]:
import time
import sqlite3
import requests
import spacy
import wikipediaapi

print("Loading scispaCy model...")
nlp = spacy.load("en_core_sci_lg")   # broader entity coverage than bc5cdr

try:
    from scispacy.abbreviation import AbbreviationDetector
    nlp.add_pipe("abbreviation_detector")
    HAS_ABBR = True
except Exception as e:
    print("AbbreviationDetector not available:", e)
    HAS_ABBR = False

wiki = wikipediaapi.Wikipedia(
    language="en",
    user_agent="MedicalGlossaryProject/1.0 (student_project)"
)


C:\Users\Ilinca\miniconda3\envs\nlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading scispaCy model...


C:\Users\Ilinca\miniconda3\envs\nlp\Lib\site-packages\spacy\language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [25]:
DB_PATH = "glossary_cache.sqlite"

def cache_init(db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS defs (
            key TEXT PRIMARY KEY,
            value TEXT
        )
    """)
    conn.commit()
    return conn

CONN = cache_init()

def cache_get(key, conn=CONN):
    cur = conn.cursor()
    cur.execute("SELECT value FROM defs WHERE key=?", (key,))
    row = cur.fetchone()
    return None if row is None else row[0]

def cache_set(key, value, conn=CONN):
    cur = conn.cursor()
    cur.execute("INSERT OR REPLACE INTO defs(key, value) VALUES (?, ?)", (key, value))
    conn.commit()


In [26]:
import re
from wordfreq import zipf_frequency

MEDICAL_SUFFIXES = (
    "itis","emia","osis","opathy","oma","ectomy","otomy","algia","genic","lysis",
    "card","cardio","neuro","hepato","nephro","derm","pulmo","renal","onc",
    "immun","therap","toxic","metabol","gastro","osteo","myo","endo","hemo","hemat"
)

MEDICAL_PREFIXES = (
    "anti","hyper","hypo","brady","tachy","intra","inter","peri","post","pre",
    "sub","trans","micro","macro","poly","mono","neo","bio","cyto","histo",
    "neuro","cardio","hemo","hemat","derm","gastro","hepato","nephro","pulmo","osteo"
)

MEDICAL_SUBSTRINGS = (
    "chemo","radio","placebo","random","double-blind","biomarker","antibody",
    "infection","bacter","viral","fung","transfus","apheres","antibiot","antifung",
    "endpoint","efficacy","adverse","toxicity","mortality","morbidity"
)

KNOWN_NON_MEDICAL = {
    "daily","day","days","week","weeks","month","months","year","years",
    "patient","patients","study","trial","group","groups","arm","arms",
    "receive","receives","received","treatment","duration","oral","dose","doses"
}


In [38]:
def normalize_term(t: str) -> str:
    t = t.strip()
    t = re.sub(r"\s+", " ", t)
    return t

def is_candidate_term(term: str) -> bool:
    if not term:
        return False
    if len(term) < 3:
        return False
    # reject mostly numeric
    if re.fullmatch(r"[\d\W_]+", term):
        return False
    # reject pure units / trivial tokens
    if term.lower() in {"mg", "ml", "cm", "mm", "kg", "bid", "tid", "qid", "prn"}:
        return False
    return True

def is_probably_medical_word(word: str, zipf_thresh=4.5) -> bool:
    """
    Fallback heuristic for SINGLE TOKENS not found by NER:
    - must be rare-ish (Zipf below threshold) OR be an uppercase acronym
    - must pass blacklist
    - must have medical-looking morphology (prefix/suffix/substring)
    """
    if not word:
        return False

    w = word.strip()
    wl = w.lower()

    # Skip boilerplate / obvious non-medical
    if wl in KNOWN_NON_MEDICAL:
        return False

    # Allow acronyms (COPD, CNS, MI)
    if w.isupper() and w.isalpha() and 2 <= len(w) <= 8:
        return True

    # Must be a simple word token (letters or hyphen)
    if not re.fullmatch(r"[a-z][a-z\-]+", wl):
        return False

    # Rarity filter: if it's common English, don't treat it as medical fallback
    if zipf_frequency(wl, "en") >= zipf_thresh:
        return False

    if any(wl.endswith(s) for s in MEDICAL_SUFFIXES):
        return True
    if any(wl.startswith(p) for p in MEDICAL_PREFIXES):
        return True
    if any(s in wl for s in MEDICAL_SUBSTRINGS):
        return True

    return False



In [28]:
BIOPORTAL_KEY = "8bac14aa-d666-4338-b2aa-5fd0ac6e2dcb" #created an account to get this
BIOPORTAL_BASE = "https://data.bioontology.org"
BIOPORTAL_ONTOLOGIES = ["MESH"]  # start with MeSH; you can add SNOMEDCT (license), NCIT, etc.

def bioportal_definition(term: str) -> str | None:
    if not BIOPORTAL_KEY:
        return None

    cache_key = f"bioportal::{term.lower()}"
    cached = cache_get(cache_key)
    if cached is not None:
        return cached if cached != "" else None

    headers = {"Authorization": f"apikey token={BIOPORTAL_KEY}"}
    params = {
        "q": term,
        "ontologies": ",".join(BIOPORTAL_ONTOLOGIES),
        "require_exact_match": "false",
        "require_definitions": "true",
        "pagesize": 5
    }

    try:
        r = requests.get(f"{BIOPORTAL_BASE}/search", headers=headers, params=params, timeout=20)
        if r.status_code != 200:
            cache_set(cache_key, "")
            return None

        data = r.json()
        # Look for the first hit that has a definition
        for hit in data.get("collection", []):
            defs = hit.get("definition") or []
            if defs:
                # defs can be list of strings
                definition = defs[0].strip()
                if definition:
                    cache_set(cache_key, definition)
                    return definition

        cache_set(cache_key, "")
        return None

    except Exception:
        cache_set(cache_key, "")
        return None

#wikipedia only used as fallback
def wikipedia_definition(term: str, max_sentences=2) -> str | None:
    cache_key = f"wiki::{term.lower()}"
    cached = cache_get(cache_key)
    if cached is not None:
        return cached if cached != "" else None

    try:
        page = wiki.page(term)
        if not page.exists():
            cache_set(cache_key, "")
            return None
        summary = page.summary.strip()
        if not summary:
            cache_set(cache_key, "")
            return None
        sents = summary.split(". ")
        short = ". ".join(sents[:max_sentences]).strip()
        if not short.endswith("."):
            short += "."
        cache_set(cache_key, short)
        return short
    except Exception:
        cache_set(cache_key, "")
        return None

def get_definition(term: str) -> str:
    # 1) BioPortal (MeSH) if key is available
    d = bioportal_definition(term)
    if d:
        return d

    # 2) Wikipedia fallback
    d = wikipedia_definition(term)
    if d:
        return d

    return "Definition not found."


In [29]:
def extract_terms(text: str, max_terms=40) -> list[str]:
    doc = nlp(text)

    terms = []
    seen = set()

    # A) Entities from scispaCy
    for ent in doc.ents:
        t = normalize_term(ent.text)
        if is_candidate_term(t) and t.lower() not in seen:
            seen.add(t.lower())
            terms.append(t)

    # B) Abbreviations: map "MI" -> "myocardial infarction" when detectable
    if HAS_ABBR and hasattr(doc._, "abbreviations"):
        for abbr in doc._.abbreviations:
            short = normalize_term(str(abbr))
            longf = normalize_term(str(abbr._.long_form))
            # store the short form but define by long form later
            if is_candidate_term(short) and short.lower() not in seen:
                seen.add(short.lower())
                terms.append(short)
            if is_candidate_term(longf) and longf.lower() not in seen:
                seen.add(longf.lower())
                terms.append(longf)

    # C) Optional fallback: add rare single-word jargon not already captured
    for tok in doc:
        if tok.is_stop or tok.is_punct:
            continue
        w = tok.text.strip()
        if w.lower() in seen:
            continue
        if is_probably_medical_word(w):
            seen.add(w.lower())
            terms.append(w)

    return terms[:max_terms]


In [30]:
def generate_glossary(text: str, sleep_s=0.2) -> dict[str, str]:
    glossary = {}
    terms = extract_terms(text)

    for term in terms:
        # polite rate limiting if using BioPortal
        if BIOPORTAL_KEY:
            time.sleep(sleep_s)

        definition = get_definition(term)
        glossary[term] = definition

    return glossary


In [ ]:
text_data = """
PRIMARY OBJECTIVES: I. To determine the effects of the iron-chelating agent deferasirox on changes in: neutrophil function; macrophage function; lymphocyte function.

SECONDARY OBJECTIVES: I. To determine the effect of chelation on the incidence of bacterial, viral and fungal infections documented by clinical, microbiologically-proven versus radiologically-proven criteria. II. To determine the effect of iron chelation on mortality and morbidity with incidence of the following parameters: Need for hospitalization; Duration of hospitalization; Need for ventilatory support; Need for exchange transfusion/apheresis; Need for treatment with antifungals or antibiotics for documented infections.

OUTLINE: Patients receive oral deferasirox once daily for up to 6 months or until blood counts recover in the absence of disease progression or unacceptable toxicity.
"""

gloss = generate_glossary(text_data)

for k, v in gloss.items():
    print(f"\n🔹 {k}\n   {v}")
